In [1]:

!pip install ultralytics pycocotools -q


import os, json, shutil, cv2
from tqdm import tqdm
from ultralytics import YOLO

#PATH
DATA_PATH = "/kaggle/input/datasets/samiatisha/dhaka-city-traffic-2024/Final_Dhaka_Traffic_Dataset"

TRAIN_PATH = f"{DATA_PATH}/train"
VAL_PATH = f"{DATA_PATH}/valid"
TEST_PATH = f"{DATA_PATH}/test"

#FIND JSON
def find_json(path):
    for f in os.listdir(path):
        if f.endswith(".json"):
            return os.path.join(path, f)

train_json = find_json(TRAIN_PATH)
val_json = find_json(VAL_PATH)

#COCO → YOLO
def convert_coco_to_yolo(json_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    with open(json_path) as f:
        data = json.load(f)

    images = {img["id"]: img for img in data["images"]}
    categories = data["categories"]
    cat_map = {cat["id"]: i for i, cat in enumerate(categories)}

    ann_per_image = {}
    for ann in data["annotations"]:
        ann_per_image.setdefault(ann["image_id"], []).append(ann)

    for img_id, anns in tqdm(ann_per_image.items()):
        img_info = images[img_id]
        h, w = img_info["height"], img_info["width"]
        fname = img_info["file_name"]

        label_path = os.path.join(output_dir, fname.replace(".jpg", ".txt"))

        with open(label_path, "w") as f:
            for ann in anns:
                x, y, bw, bh = ann["bbox"]

                xc = (x + bw/2) / w
                yc = (y + bh/2) / h
                bw /= w
                bh /= h

                cls = cat_map[ann["category_id"]]
                f.write(f"{cls} {xc} {yc} {bw} {bh}\n")

# convert
convert_coco_to_yolo(train_json, "/kaggle/working/train_labels")
convert_coco_to_yolo(val_json, "/kaggle/working/val_labels")

#YOLO FORMAT
def setup_yolo(src_img, src_lbl, dst):
    os.makedirs(f"{dst}/images", exist_ok=True)
    os.makedirs(f"{dst}/labels", exist_ok=True)

    for f in os.listdir(src_img):
        if f.endswith(".jpg"):
            shutil.copy(os.path.join(src_img, f), f"{dst}/images")

    for f in os.listdir(src_lbl):
        if f.endswith(".txt"):
            shutil.copy(os.path.join(src_lbl, f), f"{dst}/labels")

setup_yolo(TRAIN_PATH, "/kaggle/working/train_labels", "/kaggle/working/train")
setup_yolo(VAL_PATH, "/kaggle/working/val_labels", "/kaggle/working/val")


print("Train Images:", len(os.listdir("/kaggle/working/train/images")))
print("Train Labels:", len(os.listdir("/kaggle/working/train/labels")))


yaml = """
path: /kaggle/working

train: train/images
val: val/images

names:
  0: Bike
  1: Bus
  2: Car
  3: Cng
  4: Cycle
  5: Mini-Truck
  6: People
  7: Rickshaw
  8: Truck
"""

with open("/kaggle/working/data.yaml","w") as f:
    f.write(yaml)

#TRAIN
model = YOLO("yolov8s.pt")   

model.train(
    data="/kaggle/working/data.yaml",
    epochs=50,              
    imgsz=640,
    batch=16,
    lr0=0.001,
    optimizer="AdamW",
    augment=True,          
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1
)

# ACCURACY
metrics = model.val()

print("\n===== FINAL METRICS =====")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)


def weighted_congestion(boxes):
    weights = {
        0:1,  # Bike
        1:4,  # Bus
        2:2,  # Car
        3:2,  # CNG
        5:3,  # Mini-truck
        8:5   # Truck
    }

    score = 0
    for b in boxes:
        cls = int(b.cls[0])
        score += weights.get(cls, 0)

    if score < 10:
        return "Low"
    elif score < 25:
        return "Medium"
    else:
        return "High"

# TEST
img_path = f"{TEST_PATH}/{os.listdir(TEST_PATH)[0]}"
img = cv2.imread(img_path)

result = model(img)[0]

print("\nTraffic Level:", weighted_congestion(result.boxes))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.5 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 1569/1569 [00:00<00:00, 10165.24it/s]


Train Images: 7326
Train Labels: 7326
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_m